# OptiGuard Step 8: Neural Restoration Model Training
### Constrained-Gain Model Selection on Synthetic Raman Hyperspectral Maps

This notebook trains the **Spatial-Spectral Multi-Scale Restoration U-Net** on synthetic Raman maps, saves persistent checkpoints to Google Drive, and validates physical fidelity via the OptiGuard assurance harness.

In [ ]:
# Cell 2: Physics Gate & Baseline Keystone Test
# Hard assert: training halts immediately if physics asserts fail
print("Running full physics suite and CRLB keystone assertions...")
ret = !pytest tests/test_physics.py tests/test_baselines.py -v
print("\n".join(ret))
assert any("failed" not in line.lower() and ("passed" in line.lower()) for line in ret[-3:]), "PHYSICS GATE FAILED! Aborting training on invalid physics environment."

In [ ]:
# Cell 3: Pre-generate Synthetic Corpus (300 Datacubes)
!python scripts/generate_corpus.py \
    --seed 20260806 \
    --n 300 \
    --window 128 \
    --out /content/data/corpus

In [ ]:
# Cell 4: Train Spatial-Spectral U-Net with Constrained-Gain Model Selection
# Validates every 5 epochs; supports seamless resume from latest.pt on Drive
import os

latest_ckpt = os.path.join(RUNS_DIR, 'latest.pt')
resume_flag = f"--resume {latest_ckpt}" if os.path.exists(latest_ckpt) else ""

!python training/train.py \
    --config configs/restoration_v1.yaml \
    --data /content/data/corpus \
    --out {RUNS_DIR} \
    --select-on constrained_gain \
    --recall-floor 0.719 \
    --checkpoint-every 5 \
    {resume_flag}

In [ ]:
# Cell 5: Evaluate Best Restoration Model on Test Maps
import os
import json
import numpy as np
from optiguard.data.simulator import MapSimulator
from optiguard.eval.harness import evaluate
from optiguard.models.wrapper import load_restoration_method

best_checkpoint = os.path.join(RUNS_DIR, 'best.pt')
if not os.path.exists(best_checkpoint):
    print(f"NOTE: No checkpoint cleared the 0.719 recall floor at 1.5 CRLB. Evaluating latest model.")
    best_checkpoint = os.path.join(RUNS_DIR, 'latest.pt')

# Load test indices from baselines.json provenance metadata
test_indices = list(range(6, 12))
if os.path.exists("evidence/baselines.json"):
    with open("evidence/baselines.json", "r") as f:
        test_indices = json.load(f).get("_meta", {}).get("test_indices", test_indices)
print(f"Evaluating on test split indices: {test_indices}")

sim = MapSimulator.from_yaml("configs/simulator.yaml")
test_samples = [sim.generate(index=i) for i in test_indices]

restoration_fn = load_restoration_method(best_checkpoint, config_path="configs/restoration_v1.yaml")
res = evaluate(restoration_fn, test_samples, exposure=0.1)

print("=" * 65)
print("OPTIGUARD RESTORATION V1 EVALUATION RESULTS (0.1s exposure)")
print("=" * 65)
print(f"Effective Exposure Gain:  {res.effective_exposure_gain:.2f}x")
print(f"RMSE Center Error:       {res.rmse_center_cm1:.4f} cm^-1")
print(f"MAE Center Error:        {res.mae_center_cm1:.4f} cm^-1")
print(f"RMSE / CRLB(0.1s):       {res.rmse_over_crlb:.3f}")
print(f"False Feature Rate:      {res.false_feature_rate*100:.2f}%")
print("\nDefect Recall by Difficulty (CRLB multiples):")
for diff, rec in sorted(res.recall_by_difficulty.items()):
    print(f"  {diff:4.1f} CRLB: {rec*100:5.1f}%")
print("=" * 65)

In [ ]:
# Cell 6: Export ONNX Model & Package Complete Evidence Bundle
import os

onnx_path = os.path.join(RUNS_DIR, 'model.onnx')
best_checkpoint = os.path.join(RUNS_DIR, 'best.pt')
if not os.path.exists(best_checkpoint):
    best_checkpoint = os.path.join(RUNS_DIR, 'latest.pt')

!python scripts/export_onnx.py \
    --checkpoint {best_checkpoint} \
    --out {onnx_path} \
    --config configs/restoration_v1.yaml

# Save environment pip freeze for provenance
!pip freeze > {RUNS_DIR}/pip_freeze.txt

print(f"\nAll artifacts and evidence package saved to Drive: {RUNS_DIR}")
!ls -la {RUNS_DIR}